In [6]:
# epoch影响较大
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project/database/PDAC/PDAC_SC_filtered.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project/database/PDAC/PDAC_ST.h5ad" \
    --n_epochs 300 \
    --resolution 2 \
    --loss_type zinb \
    --beta 0.1 \
    --lambda_mmd 1.0 \
    --top_n_per_type 300 \
    --latent_dim 256 \
    --output_dir ../deconv_results/PDAC \
    --aggregation_method mean
# --precomputed_marker_file "/mnt/d/ST_Graduation_Project/SC_MAP_ST/deconv_results/PDAC/final_genes.txt"

Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 300
   Leiden resolution: 2.0
   Batch size: 256
   Epochs: 300
   Learning rate: 0.001
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 256
   Loss type: ZINB
   Lambda MMD: 1.0
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project/database/PDAC/PDAC_SC_filtered.h5ad
   SC shape: (1804, 19738)
   SC avg counts/cell: 3345.6 (from X)
   Loading ST: /mnt/d/ST_Graduation_Project/database/PDAC/PDAC_ST.h5ad
   ST shape: (428, 19738)
   Common genes: 19738
   SC final: (1804, 19738)
   ST final: (428, 19738)
Computing clusters and marker genes...
Starting clustering analysis...
Clustering results: 23 clusters
Marker genes per cluster:
   0: 300 -> 166 (after Lasso)
   1: 76 -> 76 (after Lasso)
   10: 300 -> 125 (after Lasso)
   11: 25 -> 25 (after Lasso)
   12: 300 -> 84 (after Lasso)
   13: 300 -> 58 (after Las

VAE Training:  42%|████▏     | 126/300 [00:42<00:58,  2.98epoch/s, Train=1549.8986, Recon=1543.3706, KL=65.2786, MMD=0.0188, Test=1659.2115]



⚠️ Early stopping triggered at epoch 127/300
   Best test loss: 1658.7713, Current test loss: 1659.2115
   No improvement for 20 consecutive epochs
📊 Computing cluster centers using ALL SC data (train + test)...
   ALL SC data: (1804, 1143)
   Number of clusters: 23
   Computed embeddings shape: (1804, 256)
Computing cluster centers with mean aggregation...
      Cluster 0: 187 cells (mean aggregation)
      Cluster 1: 149 cells (mean aggregation)
      Cluster 2: 83 cells (mean aggregation)
      Cluster 3: 82 cells (mean aggregation)
      Cluster 4: 82 cells (mean aggregation)
      Cluster 5: 70 cells (mean aggregation)
      Cluster 6: 55 cells (mean aggregation)
      Cluster 7: 51 cells (mean aggregation)
      Cluster 8: 48 cells (mean aggregation)
      Cluster 9: 45 cells (mean aggregation)
      Cluster 10: 38 cells (mean aggregation)
      Cluster 11: 28 cells (mean aggregation)
      Cluster 12: 132 cells (mean aggregation)
      Cluster 13: 23 cells (mean aggregation)
  

In [7]:
# λ 越大更稀疏0.1-0.5
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project/database/PDAC/PDAC_ST.h5ad" \
    --stage1_model_path "../deconv_results/PDAC/final_vae.pth" \
    --output_dir "../deconv_results/PDAC" \
    --gat_hidden_dim 512 \
    --gat_layers 4 \
    --lr 1e-4 \
    --loss_lambda_mse 0.1 \
    --loss_lambda_cosine 5 \
    --loss_lambda_pearson 5 \
    --loss_lambda_reg 0.1 \
    --loss_lambda_sparse 0.1 \
    --loss_lambda_proportion 0.1 \
    --k_spatial 10 \
    --k_celltype 20 \
    --batch_size 512 \
    --n_epochs 400 \
    --weight_threshold 0.05

Sample name: PDAC
Stage 1 model: ../deconv_results/PDAC/final_vae.pth
Output directory: ../deconv_results/PDAC
Device: cpu
Weight threshold: 0.05
Loading pretrained VAE Encoder...
   VAE architecture: 1143 -> 256
   Output type: zinb
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 1804 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/PDAC/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([23, 256])
   Cluster expressions (marker): torch.Size([23, 1143])
   Cluster expressions (all genes): 23 × 19738
   Loaded celltype mapping: 23 clusters
   Average cell counts: 3345.6
Loaded all genes list: 19738 genes
VAE Encoder loaded: 1143 -> 256
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '3', '4', '5', '6', '7', '8', '9']
Marker genes: 1143
Using Stage 1 cluster centers and expressions...
Loaded 23 clusters
Using Stage 1 pretrained cluster data
   Cluster centers: torch.Siz

GAT Training:  94%|█████████▍| 378/400 [04:25<00:15,  1.42epoch/s, Total=6.6291, Pearson=0.5544, MSE=15.5740, Cosine=0.4434, P_pat=21, M_pat=10, C_pat=21]



⚠️ Early stopping triggered at epoch 379/400
   All three core losses stopped improving:
      Pearson: best=0.5538, current=0.5544, no improvement for 21 epochs
      MSE: best=15.4678, current=15.5740, no improvement for 10 epochs
      Cosine: best=0.4428, current=0.4434, no improvement for 21 epochs
Evaluating model results...
Applying weight threshold: 0.05
   Non-zero elements: 8560 -> 2002 (20.3%)
Saving deconvolution results...
Generating deconvolution expression matrices...
   Reconstructing all gene expression...
   Saved: ../deconv_results/PDAC/PDAC_reconstructed_all_genes.csv
   Cell type composition...
   Found duplicate celltype names: ['Ductal', 'Cancer']. Merging corresponding cluster columns by summing weights.
   Columns before: 23, after merge: 6
   Saved cell composition (celltype): ../deconv_results/PDAC/PDAC_cell_composition.csv
   Saved cluster composition: ../deconv_results/PDAC/PDAC_cluster_composition.csv
   Using 1143 marker genes for reconstruction quality 